# 모듈 6. Memory PII 보호

기존 강의자료 `2. langgraph/2. Memory/1. MemorySaver.ipynb`의 MemorySaver 흐름에 개인정보 마스킹 가드레일을 추가합니다.

## A/B 테스트 구조

- **A: 기존 코드 그대로 사용** - 사용자 메시지를 그대로 MemorySaver 상태에 저장합니다.
- **B: 기존 코드에 가드레일 추가** - 상태에 저장되기 전에 전화번호, 주민등록번호, 이메일을 마스킹합니다.

In [ ]:
import re

from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph


user_message = (
    "안녕하세요. 제 이름은 김민수이고 전화번호는 010-1234-5678입니다. "
    "주민등록번호는 900101-1234567이고 이메일은 minsu@example.com입니다."
)

## 1단계: A - 기존 MemorySaver 흐름 그대로 사용

기존 MemorySaver 예제처럼 메시지를 상태에 넣고 checkpointer로 저장합니다. 이 경우 개인정보가 그대로 상태 스냅샷에 남습니다.

In [ ]:
def raw_chat_node(state: MessagesState):
    last_message = state["messages"][-1].content
    return {"messages": [HumanMessage(content=f"기억했습니다: {last_message}")]}


raw_builder = StateGraph(MessagesState)
raw_builder.add_node("chat", raw_chat_node)
raw_builder.add_edge(START, "chat")
raw_builder.add_edge("chat", END)

raw_app = raw_builder.compile(checkpointer=MemorySaver())
raw_config = {"configurable": {"thread_id": "raw-memory-demo"}}

raw_app.invoke({"messages": [HumanMessage(content=user_message)]}, config=raw_config)
raw_snapshot = raw_app.get_state(raw_config)

for message in raw_snapshot.values["messages"]:
    print(message.content)

### A안 문제점 확인

A안에서 보안 문제가 실제로 발생했는지 확인합니다.

In [ ]:
raw_text = "\n".join(message.content for message in raw_snapshot.values["messages"])
pii_diagnostic_patterns = {
    "주민등록번호": re.compile(r"(?<!\d)\d{6}-\d{7}(?!\d)"),
    "전화번호": re.compile(r"(?<!\d)01[016789]-\d{3,4}-\d{4}(?!\d)"),
    "이메일": re.compile(
        r"(?<![A-Za-z0-9._%+-])[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?![A-Za-z0-9.-])"
    ),
}
detected_pii = [name for name, pattern in pii_diagnostic_patterns.items() if pattern.search(raw_text)]

print("A안 보안 진단")
print("- MemorySaver 상태에 남은 개인정보:", detected_pii)
if detected_pii:
    print("=> 문제: 대화 메모리에 개인정보가 원문 그대로 저장되었습니다.")

## 2단계: B - 기존 MemorySaver 앞에 PII Guardrail 추가

그래프와 MemorySaver 구조는 그대로 두고, 저장될 메시지를 만들기 전에 개인정보를 마스킹합니다. Memory 보안은 저장된 뒤 지우는 것보다 저장되기 전에 줄이는 것이 중요합니다.

In [ ]:
pii_patterns = {
    "rrn": re.compile(r"(?<!\d)\d{6}-\d{7}(?!\d)"),
    "phone": re.compile(r"(?<!\d)01[016789]-\d{3,4}-\d{4}(?!\d)"),
    "email": re.compile(
        r"(?<![A-Za-z0-9._%+-])[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?![A-Za-z0-9.-])"
    ),
}


def mask_pii(text: str) -> str:
    masked = pii_patterns["rrn"].sub("[주민등록번호 마스킹]", text)
    masked = pii_patterns["phone"].sub("[전화번호 마스킹]", masked)
    masked = pii_patterns["email"].sub("[이메일 마스킹]", masked)
    return masked


def protected_chat_node(state: MessagesState):
    last_message = state["messages"][-1].content
    masked_message = mask_pii(last_message)
    remove_original_messages = [RemoveMessage(id=message.id) for message in state["messages"]]
    safe_messages = [
        HumanMessage(content=masked_message),
        HumanMessage(content=f"기억했습니다: {masked_message}"),
    ]
    return {"messages": remove_original_messages + safe_messages}


protected_builder = StateGraph(MessagesState)
protected_builder.add_node("chat", protected_chat_node)
protected_builder.add_edge(START, "chat")
protected_builder.add_edge("chat", END)

protected_app = protected_builder.compile(checkpointer=MemorySaver())
protected_config = {"configurable": {"thread_id": "protected-memory-demo"}}

protected_app.invoke({"messages": [HumanMessage(content=user_message)]}, config=protected_config)
protected_snapshot = protected_app.get_state(protected_config)

for message in protected_snapshot.values["messages"]:
    print(message.content)

### B안 개선 확인

B안에서 기존 코드 흐름은 유지하면서 보안 문제가 줄어드는지 확인합니다.

In [ ]:
protected_text = "\n".join(message.content for message in protected_snapshot.values["messages"])
remaining_pii = [name for name, pattern in pii_diagnostic_patterns.items() if pattern.search(protected_text)]

print("B안 보안 진단")
print("- MemorySaver 상태에 남은 개인정보:", remaining_pii or "없음")
print("- 마스킹 토큰 포함:", "[전화번호 마스킹]" in protected_text and "[주민등록번호 마스킹]" in protected_text)
print("=> 개선: 기존 MemorySaver 흐름은 유지하면서 저장 전 개인정보를 줄였습니다.")

## 정리

기존 MemorySaver 예제는 대화 상태를 쉽게 저장할 수 있게 해줍니다. 하지만 상태에 개인정보가 들어가면 그대로 저장될 수 있습니다. 기존 Memory 구조는 유지하되, 상태에 들어가기 전에 PII를 마스킹하는 노드를 추가해야 합니다.